# EX 01

“Lego2.csv”은 레고 세트의 출시 가격(Price, 단위 : 달러)에 대한 데이터이다. 테마와 부품 수를 이용하여 출시 가격을 예측하기 위한 단순선형회귀모형을 적합하려고 한다. 다음 물음에 답하여라. (Pieces : 부품 수, Year : 출시년도, Name : 제품명, Theme : 테마) (모든 유의성 검정에 대한 유의수준 α = 0.05 사용)

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import warnings
from IPython.display import HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import glob

warnings.filterwarnings(action = 'ignore')
pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_white' 

## (1)

부품 수를 $x$축으로 하고, 가격을 $y$축으로 하는 산점도를 그리시오. 단 각 점의 색상은 테마별로 다르게 표시하시오.

### sol

In [2]:
df = pd.read_csv('data/lego2.csv')

In [3]:
fig = df.plot(
    kind = 'scatter',
    x = 'Pieces',
    y = 'Price',
    color = 'Theme',
    opacity = 0.75,
    title = 'Price vs Pieces (colored by Theme)'
)

fig.update_layout(
    xaxis_title = 'Pieces',
    yaxis_title = 'Price (USD)',
    legend_title = 'Theme',
    width = 800,
    height = 600
)

fig.show()

---

## (2)

출시 가격을 예측하기 위한 모형을 정의하여라. (단, 범주형 변수인 테마를 처리하기 위한 가변수 설정을 포함하되, 변수 간 교호작용은 고려하지 않음.)

### sol

데이터 $i = 1,2 \dots n$에 대하여

- $Y_i = \text{Price (달러)}$
- $x_i = \text{Pieces (부품 수)}$
- Theme는 범주가 $K$개인 범주형 변수라고 하자.

기준 테마(reference themes)를 하나 정하여 (예: Theme의 첫 범주 또는 가장 빈도가 큰 범주) 이를 $T_1$이라 하고, 나머지 테마를 $T_2,\dots T_K$라고 하자.

그 때, 교호작용 없이 `Theme`를 더미변수로 포함한 선형회귀모형은

$$Y_i = \beta_0 + \beta_1x_i + \sum_{k=2}^{K} \gamma_{k}1(\text{Theme}_i = T_k) + \epsilon_i, \quad i = 1, 2\dots n$$

- 여기서 $1(\text{Theme}_i) = T_k$는 관측치 "i의 테마가 $T_k$이면, 1아니면 0"인 가변수이다.

- $\gamma_k\,\,(k\geq 2):$ `Theme`가 $T_K$일 때 절편 이동량
    - 즉, **같은 Pieces** 값에서 $T_k$ 테마는 기준 테마 $T_1$에 비해 평균 가격이 $\gamma_k$만큼 **높거나, 낮다**는 뜻이다.

> 해당 모형은 **테마별로 절편만 달리지고(Pieces에 대한 기울기 $\beta_1$는 공통)**, 즉 테마 간 `부품 가격 증가율(=기울기)`은 동일하다고 가정한다.

In [4]:
df["Theme"] = df["Theme"].astype("category")

ref_theme = df["Theme"].cat.categories[0]

X_theme = pd.get_dummies(df["Theme"], drop_first=True)

X = pd.concat([df[["Pieces"]], X_theme], axis=1)

#ref_theme, X.head()

---

## (3) 

회귀 모형을 적합하고, 추정된 회귀 계수들의 유의성 검정을 수행하시오.

### sol

`(1)` 1-(2)에서 정의했던 모형 활용

기준 테마(reference)를 `Friends`로 두고 (더미에서 제외), 다음 모형을 적합한다.

$$\epsilon_i \stackrel{\text{i.i.d.}}{\sim} \mathcal{N}(0,\sigma^2), \quad i=1,\dots n$$

$$\text{Price}_i = \beta_0 + \beta_1 \text{Pieces}_i + \gamma_2 1(\text{Theme} = \text{Harry Potter}) + \gamma_3 1(\text{Theme} = \text{Marvel Super Heroes}) + \gamma_4 1(\text{Theme} = \text{Star Wars}) + \epsilon_i$$

`(2)` 추정된 회귀계수 및 개별 유의성 검정(t-test)

각 회귀계수에 대하여

$$H_0: \text{해당 변수의 회귀계수} = 0 \quad vs \quad H_1: \text{해당 변수의 회귀계수} \neq 0$$

위 가설에 대한 양측 $t$-검정을 유의수준 $\alpha = 0.05$에서 수행한 결과이다.

(표본크기 $n = 212$, $df$(자유도) $n-(p+1) = 207$ )

| 계수                                 |     추정치 |     표준오차 |       t |  p-value | 결론(α=0.05) |
| ---------------------------------- | ------: | -------: | ------: | -------: | ---------- |
| Intercept ($\beta_0$)                | -1.32 |   1.80 | -0.73 |   0.46 | 유의하지 않음    |
| Pieces ($\beta_1$)                   |  0.10 | 0.002 | 68.35 |  < 2e-16 | **유의**     |
| Harry Potter ($\gamma_{HP}$)         |  2.85 |   3.97 |  0.72 |   0.47 | 유의하지 않음    |
| Marvel Super Heroes ($\gamma_{MSH}$) | 11.03 |   2.95 |  3.75 | 0.00 | **유의**     |
| Star Wars ($\gamma_{SW}$)            |  7.89 |   2.73 |  2.89 | 0.004 | **유의**     |


`(3)` summary

1. `Pieces`의 회귀계수 $\beta_1$는 매우 유의미하며, **부품 수가 증가할수록 평균 가격이 증가한다고 볼 수 있다.**

2. 테마 효과는 기준(Friends) 대비로 해석되며, **Marvel Super Heroes, Star Wars**는 **Friends** 대비 유의한 절편 차이가 관찰되었다.

3. **Hary Potter**는 **Friends**대비 차이가 통계적으로 유의하지 않았다.

In [5]:
import statsmodels.api as sm

## 반응변수 y 설정
X = X.astype(float)

## X에 상수항(intercept)만 추가해서 회귀모형 적합

X1 = sm.add_constant(X, has_constant="add")

y = df["Price"]
model = sm.OLS(y, X1).fit()

#print(model.summary())

---

## (4)

추정된 회귀계수의 의미를 설명하여라.

### sol

`1` $\beta_0$ (절편)

- 기준 테마 $T_1$에서, $\text{Pieces}=0$일 때의 평균 가격을 의미한다.

- 실제로 `Pieces`가 0인 제품은 거의 없으므로, $\beta_0$ 자체의 어떤 의미를 가지는 해석은 제한적이다.

`2` $\beta_1$: Pieces의 기울기

- 모든 테마에서 공통으로, `Piece`가 1증가할 때 **평균 가격이 $\beta_1$**만큼 증가한다고 해석할 수 있다.

- 교호작용을 포함하지 않았으므로, 테마별로 기울기가 달라지이 않는다는 가정을 내포한다.

`3` $\gamma_k$

- $\gamma_k$는 $T_k$일 때, 기준 테마 $T_1$에 비해 평균 가격이 얼마나 달라지는 지를 나타내는 절편의 이동량(`shift`)이다.

- 좀더 상세하게 말하면,

$$E(Y | x, \text{Theme} = T_{k}) - E(Y| x, \text{Theme} = T_1) = \gamma_k$$

- 이므로, 같은 Pieces($x$)에서 **기준 테마 대비 평균 가격 차이(달러)** 로 해석한다.

- $\gamma > 0$이면 $T_L$는 기준 테마보다 더 비싼, 반대의 경우 더 저렴함을 의미한다.

## (5)

테마가 출시 가격에 통계적으로 유의미한 영향을 미친다고 할 수 있는가?

(여러가지 근거를 바탕으로 설명하여라.)

### sol

`1` 비교할 두 모형

-  축소모형(Reduced model): Piece만 사용

$$M_{R}: \quad Y_i = \beta_0 + \beta_1 \times \text{Pieces}_i +\epsilon$$

- Full model: Pieces + Theme

$$M_{F}: \quad Y_i = \beta_0 + \beta_1 \times \text{Pieces}_i + \sum_{k=2}^{K} \gamma_{k} 1(\text{Theme}_i = T_{K})+\epsilon_i $$

`2`  검정 가설

Theme가 모형에 추가로 기여하는지를 보는 것이므로,

$$H_0: \gamma_k = 0 \quad vs \quad H_1: \text{적어도 하나의 } \gamma_k \neq 0$$

`3` 부분 F-검정 결론

- 부분 F-검정에서 p-value가 0.05 미만이면, Theme 더미들을 추가하는 것이 SSE를 유의미하게 감소시키므로 Theme는 Price에 통계적으로 유의미한 영향을 준다고 결론 내릴 수 있다.

| Model                        |  SSE (RSS) |    df |      MSE | 비교                 |
| ---------------------------- | ---------: | ----: | -------: | ------------------ |
| Reduced: Price ~ Pieces      | 57,322 |   210 | 273 |                    |
| Full: Price ~ Pieces + Theme | 52,952 |   207 | 256 |                    |
| Partial F                    |     5.6934 | (q=3) |          | p-value = 0.000916 |


$$F  = \frac{(Rss_R-RSS_F)/q}{RSS_{F}/df_{f}} = 5.369...$$

$$\text{p-value} = P(F_{3,207} \geq 5.6394) = 0.000916$$

- 결론: `p-value<0.05`이므로 `Theme`는 `Price`에 통계적으로 유의미한 영향을 미친다.

In [6]:
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm

model_full = model  

X_reduced = sm.add_constant(df[["Pieces"]].astype(float), has_constant="add")
model_reduced = sm.OLS(df["Price"], X_reduced).fit()

anova_res = anova_lm(model_reduced, model_full)
#anova_res

---

## (6)

잔차검정을 수행하고, 선형 회귀의 기본 가정을 잘 만족하는지를 확인하여라.

### sol

::: {.panel-tabset}

#### Residuals vs Fitted 

In [7]:
# 1) 잔차/적합값
resid = model.resid
fitted = model.fittedvalues

# (A) Residuals vs Fitted (Theme 색칠까지 하면 발표에 좋음)
tmp = df.loc[fitted.index, ["Theme"]].copy()
tmp["Fitted"] = fitted
tmp["Residual"] = resid

fig1 = px.scatter(tmp, x="Fitted", y="Residual", color="Theme", opacity=0.75,
                  title="Residuals vs Fitted (colored by Theme)")
fig1.add_hline(y=0, line_width=1)
fig1.update_layout(width=800, height=600)
fig1.show()

#### Normal Q-Q plot

In [8]:
import scipy.stats as st

r = np.asarray(resid)
r_sorted = np.sort(r)
n = len(r_sorted)
theo_q = st.norm.ppf((np.arange(1, n+1) - 0.5) / n)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=theo_q, y=r_sorted, mode="markers", name="Residuals"))
# 기준선(대략): 기울기=표준편차, 절편=평균
fig2.add_trace(go.Scatter(x=[theo_q.min(), theo_q.max()],
                          y=[r_sorted.mean() + r_sorted.std(ddof=1)*theo_q.min(),
                             r_sorted.mean() + r_sorted.std(ddof=1)*theo_q.max()],
                          mode="lines", name="Reference line"))
fig2.update_layout(title="Normal Q-Q Plot of Residuals",
                   xaxis_title="Theoretical Quantiles",
                   yaxis_title="Sample Quantiles",
                   width=800, height=600)

fig2.show()

In [9]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, normal_ad
from statsmodels.stats.stattools import durbin_watson, jarque_bera

bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(resid, model.model.exog)
w_lm, w_lm_p, w_f, w_f_p = het_white(resid, model.model.exog)

# 정규성: Jarque-Bera, Anderson-Darling
jb_stat, jb_p, skew, kurt = jarque_bera(resid)
ad_stat, ad_p = normal_ad(resid)

# 독립성 참고: Durbin-Watson
dw = durbin_watson(resid)

#bp_f_p, w_f_p, jb_p, ad_p, dw

from scipy.stats import shapiro

#w_stat, p_val = shapiro(model.resid)
#w_stat, p_val

:::

| 검정법              | Statistic |  p-value | Conclusion (α=0.05)          |
| ----------------- | --------: | -------: | ---------------------------- |
| Breusch–Pagan (F) |   23.49 | 5.01e-16 | 등분산성 **기각** (이분산 존재)         |
| White (F)         |   35.18 | 1.70e-34 | 등분산성 **기각** (이분산 존재)         |
| Shapiro–Wilk      | 0.82 |  5.90-e15| 정규성 **기각** (비정규)             |
| Durbin–Watson     |    1.79 |          | 2에 비교적 근접(강한 자기상관 징후는 크지 않음) |


<br>

`1` 잔차 및 Q-Q plot을 확인한 결과 이분산성은 확인되었으나, 정규성은 크게 위배되지 않는 것 처럼 보였음....

`2` 그 후에 추가적으로 정규성 검정에 활용되는 검정법들 적용했더니, 이상점 등으로 인해 정규성 가정을 기각한 것 같음.

`3` 자기상관검정인 더비왓슨 검정 결과 통계량이 `2`에 비교적 근접하여 강한 자기상관 징후는 크지 않는다고 해석할 수 있음.

`결론`: 이분산성은 유의하게 확인되었고, 정규성은 그림상 크게 문제 없어 보였지만 추가 검정에서 이상점 영향으로 기각되었다. 반면 Durbin–Watson 값이 2에 가까워 강한 자기상관 징후는 크지 않다.

---

## (7) 

이상점이 존재하는가?

### sol

In [10]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import OLSInfluence

infl = OLSInfluence(model)

stud_resid = infl.resid_studentized_internal


idx_out = np.where(np.abs(stud_resid) > 3)[0]

outlier_df = df.iloc[idx_out][["Name", "Theme", "Year", "Pieces", "Price"]].copy()
outlier_df["stud_resid"] = stud_resid[idx_out]

outlier_df = outlier_df.reindex(outlier_df["stud_resid"].abs().sort_values(ascending=False).index)

#outlier_df

In [11]:
import numpy as np
import plotly.express as px

plot_df = df.copy()
plot_df["obs"] = plot_df.index
plot_df["stud_resid"] = stud_resid
plot_df["outlier"] = np.where(np.abs(plot_df["stud_resid"]) > 3, "Outlier", "Normal")

fig = px.scatter(
    plot_df,
    x="obs",
    y="stud_resid",
    color="outlier",
    color_discrete_map={"Normal": "blue", "Outlier": "red"},  # 핵심
    hover_data=["Name", "Theme", "Year", "Pieces", "Price"],
    title="Studentized Residual Plot (Outliers in Red)"
)

fig.add_hline(y=0, line_width=1)
fig.add_hline(y=2, line_dash="dash", line_width=1)
fig.add_hline(y=-2, line_dash="dash", line_width=1)
fig.add_hline(y=3, line_dash="dot", line_width=1)
fig.add_hline(y=-3, line_dash="dot", line_width=1)


outliers_only = plot_df[plot_df["outlier"] == "Outlier"]
fig.add_scatter(
    x=outliers_only["obs"],
    y=outliers_only["stud_resid"],
    mode="markers+text",
    text=outliers_only["Name"],
    textposition="top center",
    marker=dict(size=10, color="red"),
    showlegend=False
)

fig.update_layout(width=800, height=600,
                  xaxis_title="Observation Index",
                  yaxis_title="Studentized Residual")
fig.show()

`1` 이상치 판별 기준: `studentized residual`

$$ e_i = y_i - \hat{y}_i,  \qquad  t_i = \frac{e_i}{\hat{\sigma}\sqrt{1-h_{ii}}}$$

$$|t_i| > 3 \ \Rightarrow\ \text{이상치(outlier) 후보}$$

| Obs (index) | Name             | Theme               | Year | Pieces | Price | Studentized Residual |
| ----------: | ---------------- | ------------------- | ---: | -----: | ----: | -------------------: |
|         124 | Hulkbuster       | Marvel Super Heroes | 2022 |   4,049 | 550.0 |                  8.2 |
|          95 | The Justifier    | Star Wars           | 2022 |   1,022 | 170.0 |                  3.7 |
|          52 | Republic Gunship | Star Wars           | 2021 |   3,292 | 400.0 |                  3.5 |
|         103 | The Razor Crest  | Star Wars           | 2022 |   6,187 | 600.0 |                 -3.4 |


---

## (8)

지랫대점이 존재하는가?

### sol

`1` 지렛대값(leverae)은 $H$의 대각원소 값이다.

$$H = X(X^{\top X})^{-1}X^{TT} \rightarrow h_{ii}$$

`2` 지렛대점 후보 판별 기준

$$h_{ii} > 2\bar h = 2\times \frac{p+1}{n}$$

$$p: \text{설병변수 개수}$$

`3` 값 대입 후 계산

$$h_{ii} > 2\bar h = 2\times \frac{p+1}{n} =  2\times \frac{5}{212} \approx  0.0472$$

즉,

$$h_{ii} > 0.0472 \rightarrow \text{leverage point 후보}$$

In [12]:

influence = model.get_influence()
hii = influence.hat_matrix_diag


n = int(model.nobs)
p1 = model.model.exog.shape[1]   
hbar = p1 / n
cutoff = 2 * hbar


idx_lev = np.where(hii > cutoff)[0]


lev_df = df.iloc[idx_lev][["Name", "Theme", "Year", "Pieces", "Price"]].copy()
lev_df["hii"] = hii[idx_lev]
lev_df = lev_df.sort_values("hii", ascending=False)



In [13]:

plot_df = df.copy()
plot_df["obs"] = plot_df.index
plot_df["hii"] = hii
plot_df["leverage_flag"] = np.where(plot_df["hii"] > cutoff, "Leverage", "Normal")

fig = px.scatter(
    plot_df,
    x="obs",
    y="hii",
    color="leverage_flag",
    color_discrete_map={"Normal": "blue", "Leverage": "red"},
    hover_data=["Name", "Theme", "Year", "Pieces", "Price"],
    title="Leverage Plot (h_ii)"
)

fig.add_hline(y=cutoff, line_dash="dash", line_width=1)

fig.update_layout(
    width=800,
    height=600,
    xaxis_title="Observation Index",
    yaxis_title="h_ii (Leverage)"
)

fig.show()

<br>

| Obs (index) | Name                                   | Theme               | Year | Pieces |  Price | ($h_{ii}$) |
| ----------: | -------------------------------------- | ------------------- | ---: | -----: | -----: | -------: |
|         103 | The Razor Crest                        | Star Wars           | 2022 |   6,187 | 599.99 |     0.28 |
|         148 | Hogwarts Express - Collectors' Edition | Harry Potter        | 2022 |   5,129 | 499.99 |     0.23 |
|         124 | Hulkbuster                             | Marvel Super Heroes | 2022 |   4,049 | 549.99 |     0.13 |
|         129 | Black Panther                          | Marvel Super Heroes | 2022 |   2,961 | 349.99 |     0.08 |
|          52 | Republic Gunship                       | Star Wars           | 2021 |   3,292 | 399.99 |     0.08 |
|          ... |...                       | ...           | ... |   ... | ... |     ... |

<br>

위 기준을 적용한 결과 총 25개의 `leverage point`가 관측되었음.

---

## (9)

영향점이 존재하는가?

### sol

`1` cook's Distance $D_i$ : 관측치 $i$를 제거했을 때 적합값 전체가 얼마나 변하는지...

- 만약 $D_i$가 충분히 크면 영향점 후보!

`2` 적용

$$D(i) = \frac{h_{ii}}{1-h_{ii}} \cdot \frac{1}{p+1} =\frac{r_{i}^2}{p+1}\,\,\times\,\,\frac{h_{ii} e^{2}_i}{\hat{\sigma}^2(1-h_{ii})^2}$$

따라서, 영향점 판단 기준을 다음과 같이 정의

$$D(i) \geq   F_{0.05}(p+1,\,\, n-p-1)=F_{0.05}(5, 207) \approx 2.2877$$

In [14]:
from scipy.stats import f


infl = OLSInfluence(model)
cooks_d = infl.cooks_distance[0]
n = int(model.nobs)
p = int(model.df_model)        
dfn = p + 1                    
dfd = n - p - 1               

cutoff = f.ppf(0.95, dfn, dfd)  #

idx_inf = np.where(cooks_d >= cutoff)[0]

inf_df = df.iloc[idx_inf][["Name", "Theme", "Year", "Pieces", "Price"]].copy()
inf_df["cooks_d"] = cooks_d[idx_inf]
inf_df = inf_df.reindex(inf_df["cooks_d"].sort_values(ascending=False).index)

In [15]:
inf_df

,Name,Theme,Year,Pieces,Price,cooks_d


영향점이 없음

---

## (10)

위에서 식별된 이상점, 지룃대점, 영향점을 제외하고 모형을 재적합시킨 후, 기존 모형과 비교하여라.

### sol

In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


idx_out = [124, 95, 52, 103]

idx_lev = [103, 148, 124, 129, 52, 132, 1, 143, 140, 205, 139, 207, 208, 147, 144,
           210, 206, 146, 141, 211, 209, 150, 142, 145, 149]

drop_idx = sorted(set(idx_out).union(set(idx_lev)))


df2 = df.drop(index=drop_idx).copy()


df2["Theme"] = df2["Theme"].astype("category")

df2["Theme"] = df2["Theme"].cat.set_categories(df["Theme"].cat.categories)

X2_theme = pd.get_dummies(df2["Theme"], drop_first=True)
X2 = pd.concat([df2[["Pieces"]], X2_theme], axis=1).astype(float)
X2 = sm.add_constant(X2, has_constant="add")

y2 = df2["Price"].astype(float)

model_refit = sm.OLS(y2, X2).fit()


compare = pd.DataFrame({
    "Full_coef": model.params,
    "Refit_coef": model_refit.params,
    "Diff(Refit-Full)": model_refit.params - model.params
})

stats_compare = pd.DataFrame({
    "Metric": ["nobs", "RSS(SSE)", "df_resid", "MSE", "R2", "Adj_R2"],
    "Full": [model.nobs, model.ssr, model.df_resid, model.mse_resid, model.rsquared, model.rsquared_adj],
    "Refit": [model_refit.nobs, model_refit.ssr, model_refit.df_resid, model_refit.mse_resid,
              model_refit.rsquared, model_refit.rsquared_adj]
})

#compare, stats_compare

결과 비교

| Metric    | Full model | Refit model | Comment          |
| --------- | ---------: | ----------: | ---------------- |
| n         |        212 |         186 | 26개 관측치 제거 후 재적합 |
| RSS (SSE) |  52952.736 |   22676.910 | SSE 크게 감소        |
| df(resid) |        207 |         182 |                  |
| MSE       |    255.810 |     124.598 | 오차분산 추정치 감소      |
| $R^2$     |     0.9595 |      0.9227 | Full이 더 큼        |
| $R^2_{adj}$ |     0.9587 |      0.9215 | Full이 더 큼        |


<br>

`1` 이상점·지렛대점을 제외하고 재적합한 결과, 잔차 제곱합과 MSE가 크게 감소하였다.

`2` 반면 결정계수값들은 감소하였다.

`3` 따라서 극단값 제거로 잔차 측면에서 모형의 안정성은 개선되었으나, 설명력은 다소 낮아졌다고 해석할 수 있다.

---

# EX 02

## (1)

부품 수(Pieces)와 출시 가격(Price)의 산점도를 그리되, 테마별로 회귀직선을 각각 추가하시오. (즉, 테마별 단순선형회귀모형 적합 결과를 그림)

### sol

In [17]:
fig = px.scatter(
    df,
    x="Pieces",
    y="Price",
    color="Theme",
    trendline="ols",           # Theme별로 OLS 직선 자동 적합
    opacity=0.75,
    hover_data=["Name", "Year"],
    title="Price vs Pieces with Theme-wise Regression Lines"
)

fig.update_layout(width=800, height=600,
                  xaxis_title="Pieces",
                  yaxis_title="Price (USD)")
fig.show()

---

## (2)

부품 수와 테마사이의 교호작용을 포함한 회귀모형을 정의하고, 각 회귀계수가 의미하는 바를 구체적으로 서술하시오.

### sol

$$Y_i = \beta_0 + \beta_{1}x_i + \sum_{k=2}^{K} \gamma_k D_{ik}  + \sum_{k=2}^{K}\delta_{K}(x_{i}D_{ik}) + \epsilon_i$$

$$Y_{i} = \text{price}, \quad x_i  = \text{Pieces}_i$$

$$D_{ik} = 1(\text{Theme}_i = T_{k})$$

In [18]:
X_inter = X_theme.mul(df["Pieces"], axis=0)
X_inter.columns = [f"{c}:Pieces" for c in X_inter.columns]

X_int = pd.concat([X, X_inter], axis=1)
#X_int.head()

`1` $\beta_0$ : 기준 테마 $T_1$에서 $x=0$일 때의 평균 가격(절편)

`2` $\beta_1$ : Pieces가 1 증가할 때 평균 가격 변화량

`3` $\gamma_{k}\,\, (k \geq 2)$: $x$가 동일 할 때, 테마 $T_{k}$는 기준 테마 $T_1$에 비해 절편이 $\gamma_k$만큼 이동

`4` $\delta_k\,\,(k\geq 2)$: 테마 $T_k$의 기울기가 기준 테마의 기울기 $\beta_1$에서 얼마나 달라지는지를 나타냄

$\to$ 즉, 테마 $T_k$의 Pieces 기울기는: $\beta_1 + \delta_k$

- 기준 테마 $T_1$:

$$E(Y| x, T_1) = \beta_0 + \beta_1x$$

-  테마 $T_k(k\geq 2)$: 

$$E(Y| x, T_{k}) = (\beta_0 + \gamma_k) + (\beta_1 + \delta_k)x$$

- 즉, $\delta_k$는 "테마 $T_k$의 부품당 단가(기울기)가 기준 테마($T_1$)와 얼마나 다른가"를 정량화 한다.

---

## (3)

교호작용 모형을 적합하고, 전체 모형의 유의성과 교호작용 항들의 개별 유의성을 보고하시오.

### sol

`1` 전체 모형 유의성(`F-test`)

$$F = 935.1, \qquad \text{p-value} = 3.03 \times 10^{151}$$

- $\alpha = 0.05$에서 전체 모형의 통계적으로 유의미하다.
    - 즉, `Price`를 설명하는 데 교호작용을 포함한 모형이 통계적으로 유의함

In [19]:
X_inter = X_theme.mul(df["Pieces"], axis=0)
X_inter.columns = [f"{c}:Pieces" for c in X_inter.columns]

X_int = pd.concat([X, X_inter], axis=1)

`2` 교호작용 항들의 개별 유의성(`T-test`)

$$H_0: \delta_k =0 \quad vs \quad H_1: \delta_k \neq 0$$

| Term (Interaction)           | Estimate | Std. Error |      t |  p-value | Conclusion (α=0.05) |
| ---------------------------- | -------: | ---------: | -----: | -------: | ------------------- |
| Harry Potter × Pieces        |   0.012 |     0.005 | 2.50 |   0.0131 | **유의**              |
| Marvel Super Heroes × Pieces |   0.036 |     0.005 | 7.65 | 7.88e-13 | **유의**              |
| Star Wars × Pieces           |   0.019 |     0.004 | 4.32 | 2.46e-05 | **유의**              |


- 새 교호작용 항이 모두 유의하므로, 테마에 따라 '부품 수가' 가격에 미치는 기울기가 기준 테마와 통계적으로 유의하게 다르다고 결론 낼 수 있음

In [20]:

X_inter = X_theme.mul(df["Pieces"], axis=0)
X_inter.columns = [f"{c}:Pieces" for c in X_inter.columns]

X_int = pd.concat([X, X_inter], axis=1).astype(float)
X_int = sm.add_constant(X_int, has_constant="add")


model_int = sm.OLS(df["Price"].astype(float), X_int).fit()

#model_int.fvalue, model_int.f_pvalue

#model_int.summary2().tables[1].loc[[c for c in model_int.params.index if ":Pieces" in c],
#                                  ["Coef.", "Std.Err.", "t", "P>|t|"]]

---

## (4)

특정 테마의 기울기가 기준 테마와 통계적으로 유의미하게 다른지 확인하고, 이를 “부품당 단가” 관점에서 해석하시오.

### sol

교호작용 모형에서 기준 테마(예: Friends)의 Piece 기울기를 $\beta_1$, 테마 $T_k$의 교호작용 계수를 $\delta_k$라 하면,

- 기준 테마의 기울기(부품당 단가):

$$\text{slope}(T_1) = \beta_1 $$

-  테마 $T_k$의 기울기 (부품당 단가):

$$\text{slope}(T_k) = \beta_1  + \delta_k$$

따라서 위 질문은

$$H_0: \delta_k =0 \quad vs \quad H_1: \delta_k \neq 0$$

와 동일하다. 즉, `부품당 단가` 관점에서 통계적으로 유의미하게 다르다.

---

## (5)

교호작용이 없는 모형과 교호작용이 포함된 모형에 대한 부분 F-검정(Partial F-test)을 수행하고, 어떤 모형이 본 데이터에 더 적합한지 판정하시오

### sol

`1` 비교할 두 모형

- Reduced model: 교호작용 없음

$$M_R: \quad Y = \beta_0 + \beta_1 x + \sum_{k=2}^{K} \gamma_{k} D_k +\epsilon_i $$

- Full model: 교호작용 포함

$$M_F: \quad Y = \beta_0 + \beta_1 x + \sum_{k=2}^{K} \gamma_{k} D_k + \sum_{k=2}^{K} \delta(x D_k) + \epsilon_i $$

($x = \text{pieces}, D_k = 1(\text{Theme}= T_k)$)

`2` 부분 F-검정

교호작용 항들이 `추가로` 필요한지를 검정하므로

$$H_0: \delta_2 = \dots \delta_k = 0, \quad vs \quad H_1: \text{적어도 하나의 } \delta_k \neq 0$$

`3`

| Comparison                                     | q |      RSS(_R) |      RSS(_F) |         F |      p-value | Decision (α=0.05)                |
| ---------------------------------------------- | -----------: | -----------: | -----------: | --------: | -----------: | -------------------------------- |
| $M_R \,\,\text{vs}\,\, M_F$ |            3 | 52,953 | 39,515 | 23.1 | <0.05 | 교호작용 포함 모형 채택 |


---

## (6)

적합된 최종 모형을 바탕으로 잔차 분석을 수행하고, 교호작용을 넣음으로써 이전 모형에서 나타났던 잔차의 문제점이 개선되었는지 논하시오

### sol

::: {.panel-tabset}

#### Resid vs Fitted

In [21]:
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots

def resid_fitted_df(model, df):
    fitted = model.fittedvalues
    resid = model.resid
    tmp = df.loc[fitted.index, ["Theme"]].copy()
    tmp["Fitted"] = fitted
    tmp["Residual"] = resid
    return tmp
y = df["Price"].astype(float)

# --- Reduced: Price ~ Pieces + Theme (교호작용 없음)
X_red = sm.add_constant(X.astype(float), has_constant="add")
model_red = sm.OLS(y, X_red).fit()

# --- Full: Price ~ Pieces * Theme (교호작용 포함)
X_inter = X_theme.mul(df["Pieces"], axis=0)
X_inter.columns = [f"{c}:Pieces" for c in X_inter.columns]

X_int = pd.concat([X, X_inter], axis=1).astype(float)
X_int = sm.add_constant(X_int, has_constant="add")
model_int = sm.OLS(y, X_int).fit()


tmp_red = resid_fitted_df(model_red, df)
tmp_int = resid_fitted_df(model_int, df)

fig = make_subplots(rows=1, cols=2, subplot_titles=("No interaction", "With interaction"))

fig_red = px.scatter(tmp_red, x="Fitted", y="Residual", color="Theme", opacity=0.75).data
for tr in fig_red:
    fig.add_trace(tr, row=1, col=1)

fig_int = px.scatter(tmp_int, x="Fitted", y="Residual", color="Theme", opacity=0.75).data
for tr in fig_int:
    # 범례가 너무 길면 col=2는 legend를 숨겨도 됨
    tr.showlegend = False
    fig.add_trace(tr, row=1, col=2)

fig.add_hline(y=0, line_width=1, row=1, col=1)
fig.add_hline(y=0, line_width=1, row=1, col=2)

fig.update_layout(width=900, height=500, title="Residuals vs Fitted: Model Comparison")
fig.show()

#### Q-Q plot

In [22]:

def qq_data(resid):
    r = np.asarray(resid)
    r_sorted = np.sort(r)
    n = len(r_sorted)
    theo_q = st.norm.ppf((np.arange(1, n+1) - 0.5) / n)
    return theo_q, r_sorted

theo_r, rs_r = qq_data(model_red.resid)
theo_i, rs_i = qq_data(model_int.resid)

figqq = make_subplots(rows=1, cols=2, subplot_titles=("No interaction", "With interaction"))

figqq.add_trace(go.Scatter(x=theo_r, y=rs_r, mode="markers", name="Residuals"), row=1, col=1)
figqq.add_trace(go.Scatter(x=theo_i, y=rs_i, mode="markers", showlegend=False), row=1, col=2)

figqq.update_layout(width=1200, height=500, title="Normal Q-Q Plot: Model Comparison",
                    xaxis_title="Theoretical Quantiles", yaxis_title="Sample Quantiles", showlegend = False)
figqq.show()

:::

`1` 교호작용을 포함한 모형에서는 잔차 plot의 패턴이 이전보다는 완화되었지만..... 추가 작업이 필요해보임, 여전히 이분산성이 관측됨.

`2` Q-Q plot의 경우에도 양측 꼬리의 패턴이 기존보다는 완화 되었지만, 첫 번째 잔차값이 정규성에 위해?를 가하는 이상치로 판단됨.

---

## (7)

이상치, 지룃대점, 영향점이 존재하는 지를 확인하고, 존재한다면 이 값들을 제외하고 모형 적합을 한 후 기존 모형과 비교하여라.

### sol

`1` 이상치: $|t_i| > 3$이면 이상치 후보

`2` 지렛댇점:

$$h_ii > 2\bar h, \quad \bar h = \frac{p+1}{n}$$

`3` 영향점

$$D_i \geq F(0.05, P+1, n-p-1)$$

In [23]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import f
from statsmodels.stats.outliers_influence import OLSInfluence

# 0) influence 객체
infl = OLSInfluence(model_int)

# 1) Outlier: studentized residual
t_i = infl.resid_studentized_internal
idx_out = np.where(np.abs(t_i) > 3)[0]

# 2) Leverage: h_ii > 2*(p+1)/n  (p+1 = exog cols)
hii = infl.hat_matrix_diag
n = int(model_int.nobs)
p1 = model_int.model.exog.shape[1]
cut_lev = 2 * (p1 / n)
idx_lev = np.where(hii > cut_lev)[0]

# 3) Influence: Cook's D >= cutoff
cooks_d = infl.cooks_distance[0]
dfn = p1                  # = p+1
dfd = n - (p1)            # = n-(p+1) = df_resid
cut_inf = f.isf(0.05, dfn, dfd)   # 우측 0.05 (== ppf(0.95))
idx_inf = np.where(cooks_d >= cut_inf)[0]

# 4) 제거 인덱스 합집합
drop_idx = sorted(set(idx_out) | set(idx_lev) | set(idx_inf))

#print("Outliers:", len(idx_out))
#print("Leverage points:", len(idx_lev))
#print("Influential points:", len(idx_inf))
#print("Total removed:", len(drop_idx))

# 5) 제거 후 데이터
df2 = df.drop(index=df.index[drop_idx]).copy()

# 6) 동일한 방식으로 design matrix 재구성 (Pieces*Theme)
df2["Theme"] = df2["Theme"].astype("category")
df2["Theme"] = df2["Theme"].cat.set_categories(df["Theme"].cat.categories)

X_theme2 = pd.get_dummies(df2["Theme"], drop_first=True)
X2 = pd.concat([df2[["Pieces"]], X_theme2], axis=1)

X_inter2 = X_theme2.mul(df2["Pieces"], axis=0)
X_inter2.columns = [f"{c}:Pieces" for c in X_theme2.columns]

X_int2 = pd.concat([X2, X_inter2], axis=1).astype(float)
X_int2 = sm.add_constant(X_int2, has_constant="add")

y2 = df2["Price"].astype(float)

model_int_refit = sm.OLS(y2, X_int2).fit()

# 7) 비교 지표
stats_compare = pd.DataFrame({
    "Metric": ["nobs", "RSS(SSE)", "df_resid", "MSE", "R2", "Adj_R2"],
    "Original": [model_int.nobs, model_int.ssr, model_int.df_resid, model_int.mse_resid,
                 model_int.rsquared, model_int.rsquared_adj],
    "Refit": [model_int_refit.nobs, model_int_refit.ssr, model_int_refit.df_resid,
              model_int_refit.mse_resid, model_int_refit.rsquared, model_int_refit.rsquared_adj]
})
#stats_compare

`1` 이상치/지렛대점/영향점 존재 여부 요약

| Type                  | n |  
| ----------------| :---:| 
| Outlier               | 5 |
| Leverage              |  8 |   
| Influence             | 2 |    
| Total removed (union) | 9 |  


`2` 모형 비교

| Metric    | 기존 교호작용 모델| 이상치 등 제거 모델 | Comment    |
| --------- | :------: | :-------: | ---------- |
| n         |                        212 |                 203 | 9개 관측치 제거  |
| RSS(SSE)  |                  39,515|           19,889 | 잔차 제곱합 감소  |
| MSE       |                    19 |             102 | 평균 잔차 제곱합 감소  |
| $R^2$     |                     0.97 |              0.93 | 설명력 지표는 감소 |
| $R^2_{adj}$ |                     0.97 |              0.93 | 설명력 지표는 감소 |

`3` 결과 요약

- 이상치·지렛대점·영향점 후보를 제거하고 교호작용 모형을 재적합한 결과, `RSS`와 `MSE`가 크게 감소하여 잔차 변동 측면에서는 개선되었다.

- 반면 결정계수 값들은 오히려 감소되었는데, 이는 각 이상치, 지렛대점, 영향치 후보군들을 제거할 경우, 모형의 적합 안정성을 높이는 대신 데이터 변동을 설명하는 정도는 다소 낮아진다고 볼 수 있다.

---

## (8)

만약 어떤 테마는 부품 수가 적을 때는 저렴하지만 부품 수가 많아질수록 가격이 급격히 비싸진다면, 이는 교호작용 계수의 부호와 크기에 어떤 영향을 주겠는가? 데이터의 분포와 연결 지어 발표하시오.

### sol

In [24]:
fig = px.scatter(
    df,
    x="Pieces",
    y="Price",
    color="Theme",
    opacity=0.75,
    hover_data=["Name", "Year"],
    title="Price vs Pieces with Theme-wise Regression Lines"
)

fig.update_layout(width=800, height=500,
                  xaxis_title="Pieces",
                  yaxis_title="Price (USD)")
fig.show()

`1` 전반적으로 Pieces가 증가할수록 Price가 증가하는 양(+)의 관계를 보인다.

`2` 그림에서 `Star Wars`와 `Marvel Super Heros`는 2,000 이상의 Pieces 영역에서 가격이 상대적으로 더 높게 나타나는 점들이 다수 존재하여, 부품 수가 커질수록 가격이 더 급격히 증가하는 패턴을 보인다.

`3` 따라서 이러한 분포는 교호작용 계수가 양으로 추정되도록 만들며, 특히 큰 Pieces 구간에서 기존 테마와의 가격 격차가 더 빠르게 벌어질 수록 교호작용 계수가 커져 "부품당 단가(기울기)"가 더 급격히 증가한다.

---

# EX 03

피노 누아 와인의 품질(Quality)은 투명도, 향, 바디, 맛, 오크향의 특성 그리고 지역과 관련이 있다고 생각된다. R의 MPV 패키지에 내장된 피노 누아 와인 데이터셋인 table.b11을 활용한다. Flavor와 Region
변수를 이용하여 품질을 예측하는 회귀 모형을 적합하려고 한다. 다음 물음에 답하여라.

## (1)

MPV 패키지를 설치하고 table.b11 데이터를 로드하시오. 종속변수를 Quality(y)로, 독립변수를 Flavor(x)와 Region(d)으로 설정하고 데이터의 구조(str())를 확인하시오. 현재 Region 변수가 어떤
자료형으로 저장되어 있는지 기술하시오

### sol

In [25]:
df_wine = pd.read_csv("data/table.b11.csv") 
#df_wine.info()

`1` 데이터는 총 $n=38$개 관측치이고, 7개 변수(Clarity, Aroma, Body, Flavor, Oakiness, Quality, Region)로 구성되어 있다.

`2` 이 중 Region 변수는 int64(정수형)으로 저장되어 있다.

`3` 따라서 Region을 그대로 회귀모형에 넣으면, 범주형이 아니라 수치형(선형) 변수처럼 처리되어 “Region이 1 증가할 때 Quality가 일정량 변한다”는 형태의 모형(Model A)이 된다.

---

## (2)

(Model A) Region 변수가 정수형(Integer: 1, 2, 3)으로 입력되어 있는 상태 그대로 회귀 모형을 적합하려고 한다.


a. 품질을 예측하기 위한 모형을 정의하여라.

b. 회귀모형을 적합하고, 모형의 유의성 및 회귀계수의 유의성을 검정하여라.

c. 추정된 회귀계수의 통계적 의미를 설명하시오.

d. 이 모델이 가정하는 “지역 번호와 품질 사이의 관계”가 실제 데이터 분석에서 타당한 가설인지 논하시오.

### sol

#### a. 모형 정의

$y= \text{Quality}, \, x= \text{Flavor}, \,d = \text{Region}$

$$y = \beta_0 + \beta_1x + \beta_2d + \epsilon$$

#### b. 모형 적합후, 유의성 검정

`1` 모형 유의성 검정

| Test           | H0                  | Statistic | p-value | Conclusion ($\alpha=0.05$) |
| -------------- | ------------------- | --------: | ------: | ------------------- |
| F-test | $(\beta_1=\beta_2=0)$ |      ($F=31.06$) |    1.74e-08 |귀무가설 기각 |

`2` 개별 회귀계수 유의성 검정

| Coefficient           | Estimate | Std. Error |        t |      p-value | Conclusion (α=0.05) |
| --------------------- | -------: | ---------: | -------: | -----------: | ------------------- |
| Intercept ($\beta_0$)| 5.0 |   0.985| 5.08 | 1.26e-05 | **유의**              |
| Flavor ($\beta_1$) | 1.42|   0.234 | 6.10 | 5.77e-07 | **유의**              |
| Region ($\beta_2$)| 0.34 |   0.275 | 1.23 | 2.28-01 | 유의하지 않음             |


In [26]:
import statsmodels.api as sm

y = df_wine["Quality"].astype(float)
X = df_wine[["Flavor", "Region"]].astype(float)
X = sm.add_constant(X, has_constant="add")

model_A = sm.OLS(y, X).fit()

#model_A.fvalue, model_A.f_pvalue
#model_A.summary2().tables[1][["Coef.","Std.Err.","t","P>|t|"]]

#### c. 추정된 회귀계수의 통계적 의미.

$$E(\text{Quality}\,|\,\text{Flavor} = x, \text{Region} = d) = \beta_0 + \beta_1x +\beta_2 d$$

- $\hat\beta_0=5.0039$ (**유의**): $x=0, d=0$에서의 기준 평균(해석은 기준점 성격이 큼).
- $\hat\beta_1=1.4267$ (**유의**): Region을 고정하면 Flavor가 1 증가할 때 Quality의 평균이 약 1.43 증가.
- $\hat\beta_2=0.3372$ (유의하지 않음): Flavor를 고정하면 Region이 1 증가할 때 Quality 평균이 약 0.34 증가로 추정되지만, 통계적으로 유의하지 않음($p=0.228$).

#### d. 지역 번호와 품질 사이의 관계

`1` Model A는 Region(1,2,3)을 숫자로 넣어서 “지역 번호가 1 증가할 때마다 Quality가 일정하게(선형·등간으로) 변한다”는 가정을 한다.  

`2` 하지만 Region은 본질적으로 범주형이므로 이런 선형 가정은 강한 제약이고, 실제로는 Region을 범주형으로 처리한 모형(Model B)과 비교해서 판단하는 것이 더 타당하다.

---

## (3)

(Model B) Region을 범주형 변수(as.factor)로 변환하여 새로운 회귀 모형을 적합하시오.

a. 품질을 예측하기 위한 모형을 정의하여라.

b. 회귀모형을 적합하여고, 모형의 유의성 및 회귀계수의 유의성을 검정하여라.

c. 각 지역별로 추정된 회귀계수가 의미하는 바를 (2)번 모델의 결과와 비교하여 서술하시오.

### sol

#### a. 모형 정의

- 반응변수는 `Quality`, 설명변수는 `Flavor`이며, `Region`은 숫자가 아니라 **범주형 변수**로 처리한다.  

- Region이 3개 범주(1,2,3)이므로 기준지역을 `Region=1`로 두면, Model B는 다음과 같이 정의할 수 있다.

$$\text{Quality}_i = \beta_0+\beta_1\,\text{Flavor}_i + \gamma_2\,\mathbf{1}(\text{Region}_i=2) + \gamma_3\,\mathbf{1}(\text{Region}_i=3) + \epsilon_i,
\quad \epsilon_i \sim i.i.d.\ \mathcal{N}(0,\sigma^2)$$

- $\beta_0$: 기준지역(Region=1)에서 Flavor=0일 때의 평균 Quality(절편)  
- $\beta_1$: Flavor가 1 증가할 때 Quality 평균의 변화량(지역 공통 기울기)  
- $\gamma_2$: `Region=2`가 Region=1 대비 평균 Quality가 얼마나 다른지 
- $\gamma_3$: `Region=3`이 Region=1 대비 평균 Quality가 얼마나 다른지

#### b. 회귀모형 적합, 유의성 검증

Model B는 `Region`을 범주형으로 처리한 회귀모형이다. 따라서 회귀계수의 유의성은 다음 두 수준에서 확인한다.

`1` 전체 모형 유의성 검증

- 귀무가설: Flavor 및 Region(더미항)이 모두 필요 없다.  $\to \,\,H_0: \beta_1=\gamma_2=\gamma_3=0$
- 대립가설: 적어도 하나는 0이 아니다.  
- F 통계량과 p-value를 보고, p-value < 0.05이면 “모형은 유의”라고 결론낸다.

| Test | Statistic | p-value | Conclusion (α=0.05) |
|---|---:|---:|---|
| F-test | 53.13  | 6.35e-13|귀무가설 기각|


`2` 개별 회귀계수 유의성 검증

| Term | Estimate | Std. Error | t | p-value | Conclusion (α=0.05) |
|---|---:|---:|---:|---:|---|
| Intercept | 7.09|0.79|  8.97|1.76e-10|귀무가설 기각|
| Flavor |  -1.53|0.37|-4.16|2.05e-04|귀무가설 기각|
| Region=2 (vs 1) | 1.22|0.40|3.06|4.35e-03|귀무가설 기각|
| Region=3 (vs 1) |1.11|0.17|6.42|2.49e-07|귀무가설 기각|

In [27]:
import statsmodels.formula.api as smf

model_B = smf.ols("Quality ~ Flavor + C(Region)", data=df_wine).fit()

#model_B.fvalue, model_B.f_pvalue

#model_B.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]]

#### c. Model A vs B

Model B에서 `Region`은 범주형이므로, 지역 효과는 “번호가 1 증가할 때”가 아니라 **기준지역(Region=1) 대비 평균 수준(절편)의 차이**로 해석한다.

- Intercept: 기준지역(Region=1)에서 Flavor=0일 때의 평균 Quality(기준점).
- Flavor 계수: Region을 고정했을 때 Flavor가 1 증가하면 평균 Quality가 얼마나 변하는지(지역 공통 기울기).
- Region=2 더미 계수: Region=2의 평균 Quality가 기준지역(Region=1)보다 얼마나 높은지/낮은지(절편 차이).
- Region=3 더미 계수: Region=3의 평균 Quality가 기준지역(Region=1)보다 얼마나 높은지/낮은지(절편 차이).

즉, Model B는 지역 간 차이를 “선형 등간(1→2→3)”으로 강제하지 않고, 각 지역의 평균 수준 차이를 더 일반적으로 허용한다.

---

## (4)

(Model C) Region과 Flavor 사이의 교호작용(Interaction)을 포함한 최종 모형을 적합하시오. 이 모델이 (3)번 모델에 비해 통계적으로 유의미하게 향상되었는지 anova() 함수를 통한 부분 F-검정
(Partial F-test)으로 증명하시오.


### sol

`1` 두 모형 정의

- **Model B (Reduced; 교호작용 없음)**  
  Region은 범주형으로 포함하되, Flavor의 기울기는 모든 Region에서 동일하다고 가정한다.  
  - 모형식: `Quality ~ Flavor + C(Region)`

- **Model C (Full; 교호작용 포함)**  
  Region별로 절편뿐 아니라 Flavor의 기울기까지 달라질 수 있도록 교호작용을 포함한다.  
  - 모형식: `Quality ~ Flavor * C(Region)`

  (여기서 `*`는 main effect + interaction을 모두 포함)

`2` 부분 F-검정

- **귀무가설 $H_0$**: Region×Flavor 교호작용 효과가 없다  
  (즉, 모든 교호작용 계수들이 0)
  $H_0:\ \delta_2=\delta_3=\cdots=0$
  

- **대립가설 $H_1$**: 적어도 하나의 교호작용 계수는 0이 아니다
  $H_1: \text{ at least one } \delta_k \neq 0$

`3` 결론

| Comparison         | df_diff | RSS(B) | RSS(C) |  F | p-value | Decision (α=0.05) |
| ------------------ | ------: | -----: | -----: | -: | ------: | ----------------- |
| Model B vs C |2|27.21|25.42|1.12|0.33|귀무가설 기각 x|


In [28]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# Model B (Reduced)
model_B = smf.ols("Quality ~ Flavor + C(Region)", data=df_wine).fit()

# Model C (Full: interaction 포함)
model_C = smf.ols("Quality ~ Flavor * C(Region)", data=df_wine).fit()

# Partial F-test
anova_res = anova_lm(model_B, model_C)
#anova_res

## (5)

통계적 지표와 해석의 용이성을 모두 고려했을 때, 본인이 와인 비평가라면 어떤 모형을 최종적으로 제안하겠는가? 본인의 논리적 근거를 바탕으로 발표하시오.

### sol

`1` **Model B(Region을 범주형으로 포함, 교호작용 없음)** 를 최종 모형으로 제안한다.  


`2` 이유는 `Model C`의 교호작용(Region×Flavor)은 부분 F-검정에서 유의하지 않아 복잡한 교호작용을 추가할 통계적 근거가 부족하다.

`3`  또한, Model B는 지역 효과를 `번호의 선형 증가`로 가정하지 않고 `지역별 평균 차이`로 해석할 수 있어 해석이 더 자연스럽기 때문이다.

`4` **따라서 통계적 지표와 해석 용이성을 함께 고려하면 Model B가 가장 합리적인 선택이다.**